#### In this notebook, we take the data frame created of the player match data and merge it with meta data of competition. Meta data of competition refers to categories created for each unique competition such as club league or others, gender, state or national, which state. Basically, the player-match data frame is categorised here.

In [2]:
from pathlib import Path
import pandas as pd
import json

In [3]:
#Getting success records

# read project folder (1 folder up from where we are right now)
data_collection = Path("../data_collection")

# read scrape log
log = pd.read_csv(data_collection / "scrape_log.csv")

#keeping only successful logs
log_success = log[log["status"] == "success"]

# getting file NAMES from the successful logs in a list [] called success_files
success_files = log_success["file_name"].tolist()

# we have the file names of success files. Now we want the actual files, not just the names. For that, we need to define the path for python.
json_folder = data_collection / "raw_match_centres" # note that raw_match_centre is where all your downloaded JSONs are from your scraper

# getting success files and saving them in a list called records
records = []

for file in success_files: #looping on each file in success_files 
    with open(json_folder / file, "r", encoding="utf-8") as f: #adding them here
        records.append(json.load(f))

# sanity check 
len(records)
# 21,027 success records.

21027

In [4]:
#creating empty rows for player-match dataset
player_rows = []

#labelling the data we need for each JSON in records
for record in records: #loop through each JSON (biggest loop)
    
    # READABILITY
    
    # Give shorter names to the relevant parts of the JSON
    fixture = record["fixture"] #now fixture dictionary in each JSON is called fixture
    tournament = record["tournament"] #now events list is called events in each JSON
    events = record["events"] #now tournament dictionary is called tournament

    # Combine both teams' player lists
    players = (
        record["team1_players"]
        + record["team2_players"]
    )

    # SUBSTITUTION LOGIC
    
    # Loop through every player listed for this match/this JSON
    for player in players:

        # If any player information is empty, flag it!
        t1_empty = len(record["team1_players"]) == 0
        t2_empty = len(record["team2_players"]) == 0
        one_team_empty = t1_empty != t2_empty

        # Start by assuming the player did not come on as a substitute
        substituted = False

        # Check every event in this match
        for event in events:

            # A type 4 event is a substitution.
            # sub_aiff_player_id is the player who came on.
            if (
                event.get("type") == 4
                and event.get("sub_aiff_player_id") == player.get("aiff_id")
            ): #if type = 4 and id is there,
                substituted = True
                break #if you find a match of substitution for this player, break this loop, don't check more events, go to appearance loop!

        # Build one player-match row
        row = {
            "match_id": fixture.get("id"),
            "competition_id": fixture.get("tournament_id"),
            "competition_name": fixture.get("tournament_name"),
            "match_date": fixture.get("date"),
            "match_venue": fixture.get("venue"),
            "match_stadium": fixture.get("stadium"),
            "ft": fixture.get("ft"),
            "ft_text": fixture.get("ft_text"),

            "team_type": tournament.get("team_type"),

            "player_name": player.get("player_name"),
            "player_id": player.get("player_id"),
            "player_aiff_id": player.get("aiff_id"),
            "player_team_id": player.get("team_id"),
            "status": player.get("status"),
            "one_team_player_data_missing": one_team_empty,

            "substituted": substituted,
        }

        # Add this row to the list
        player_rows.append(row)

# Convert all rows into a dataframe
player_matches = pd.DataFrame(player_rows)

In [8]:
# getting csv with competition meta data
comp_categories = pd.read_csv("competitions_classified_AI_Human.csv")

In [9]:
comp_categories.head(10)

,competition_name,player_rows,cum_pct,AI_level,AI_state,AI_age,AI_primary_format_teams,AI_gender
0,AIFF Elite Youth League 2025-26,31990,3.962263,National,National,Youth,Club League,Male
1,AIFF Sub Junior League 2025-26,24043,6.940214,National,National,Youth,Club League,Male
2,AIFF Junior League 2025-26,21229,9.569626,National,National,Youth,Club League,Male
3,Calcutta Football League - Premier Division 20...,16073,11.560418,State,West Bengal,Senior,Club League,Male
4,Reliance Foundation Development League 2025-26,15929,13.533375,National,National,Youth,Club League,Male
5,Calcutta Football League - 1st Division 2025-26,10797,14.870685,District,West Bengal,Senior,Club League,Male
6,KSFA Men's Super Division League 2025-26,10416,16.160804,State,Karnataka,Senior,Club League,Male
7,GOA PROFESSIONAL LEAGUE 2025-26,9086,17.286191,State,Goa,Senior,Club League,Male
8,Calcutta Football League 4th Division 2025-26,8761,18.371323,District,West Bengal,Senior,Club League,Male
9,Poona District Football Association- Super Div...,7944,19.355262,District,Maharashtra,Senior,Club League,Male


In [10]:
player_matches.head(10)

,match_id,competition_id,competition_name,match_date,match_venue,match_stadium,ft,ft_text,team_type,player_name,player_id,player_aiff_id,player_team_id,status,one_team_player_data_missing,substituted
0,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Kabir Mehta,492465,418848,19665,1,False,True
1,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Tavish Sharma,492473,398082,19665,2,False,False
2,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Eklavya Choudhary,496786,522296,19665,3,False,False
3,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Swastik Sharma,492474,398260,19665,1,False,False
4,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Navya Mittal,492468,418674,19665,1,False,False
5,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Aahil Khan,499647,527453,19665,1,False,False
6,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Aaradhy Bairathi,492459,418389,19665,1,False,True
7,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Ishdeep Singh,492464,418852,19665,1,False,True
8,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Raghav Vaswani,495763,418291,19665,1,False,True
9,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Hriday Chandani,494704,276078,19665,2,False,False


In [11]:
# merging datasets into a new dataset 

# Keep only the metadata columns needed for analysis from the competition metadata dataset
competition_metadata = comp_categories[
    [
        "competition_name",
        "AI_level",
        "AI_state",
        "AI_age",
        "AI_primary_format_teams",
        "AI_gender"
    ]
]

# Merge competition metadata onto player-match dataset and create new dataset called player_matches_enriched
player_matches_enriched = player_matches.merge(
    competition_metadata,
    on="competition_name",
    how="left",
    validate="many_to_one"
)

# Check merge
print(player_matches_enriched.shape)
player_matches_enriched.head()

(807367, 21)


,match_id,competition_id,competition_name,match_date,match_venue,match_stadium,ft,ft_text,team_type,player_name,...,player_aiff_id,player_team_id,status,one_team_player_data_missing,substituted,AI_level,AI_state,AI_age,AI_primary_format_teams,AI_gender
0,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Kabir Mehta,...,418848,19665,1,False,True,State,Rajasthan,Youth,Club League,Male
1,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Tavish Sharma,...,398082,19665,2,False,False,State,Rajasthan,Youth,Club League,Male
2,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Eklavya Choudhary,...,522296,19665,3,False,False,State,Rajasthan,Youth,Club League,Male
3,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Swastik Sharma,...,398260,19665,1,False,False,State,Rajasthan,Youth,Club League,Male
4,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Navya Mittal,...,418674,19665,1,False,False,State,Rajasthan,Youth,Club League,Male


In [12]:
len(player_matches_enriched['match_id'].value_counts())

18205

In [36]:
player_matches_enriched.to_csv("scraped_categorised_data.csv")

### Some Exploratory Analysis (because I am excited!):

In [35]:
# what percentage of our entire dataset (excluding where both players missing) is youth league v/s senior league?
player_matches_enriched['AI_age'].value_counts(normalize=True)*100
# equal - equal!

AI_age
Senior     50.051037
Youth      49.818798
Unclear     0.130165
Name: proportion, dtype: float64

In [16]:
# what percentage of our entire dataset (excluding where both players missing) is state league v/s national league?
player_matches_enriched['AI_level'].value_counts(normalize=True)*100
# 60% state (reason why you can't leave state), 22% national, 18% district.

AI_level
State            59.778914
National         22.016360
District         18.023385
Unclear           0.130165
International     0.051176
Name: proportion, dtype: float64

In [18]:
# what percentage of our entire dataset (excluding where both players missing) is club league v/s other formats?
player_matches_enriched['AI_primary_format_teams'].value_counts(normalize=True)*100
# 87% are club leagues!

AI_primary_format_teams
Club League    87.771872
Other          12.097963
Unclear         0.130165
Name: proportion, dtype: float64

In [20]:
# what percentage of our entire dataset (excluding where both players missing) is female v/s male?
player_matches_enriched['AI_gender'].value_counts(normalize=True)*100
# 90% are male games!

AI_gender
Male       90.177419
Female      9.692416
Unclear     0.130165
Name: proportion, dtype: float64

In [21]:
# creating analysis dataset with Club Leagues, Male, Youth
male_youth_club_player_matches = player_matches_enriched[
    (player_matches_enriched["AI_primary_format_teams"] == "Club League") &
    (player_matches_enriched["AI_gender"] == "Male") &
    (player_matches_enriched["AI_age"] == "Youth")
].copy()

print(male_youth_club_player_matches.shape)
male_youth_club_player_matches.head()

(306894, 21)


,match_id,competition_id,competition_name,match_date,match_venue,match_stadium,ft,ft_text,team_type,player_name,...,player_aiff_id,player_team_id,status,one_team_player_data_missing,substituted,AI_level,AI_state,AI_age,AI_primary_format_teams,AI_gender
0,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Kabir Mehta,...,418848,19665,1,False,True,State,Rajasthan,Youth,Club League,Male
1,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Tavish Sharma,...,398082,19665,2,False,False,State,Rajasthan,Youth,Club League,Male
2,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Eklavya Choudhary,...,522296,19665,3,False,False,State,Rajasthan,Youth,Club League,Male
3,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Swastik Sharma,...,398260,19665,1,False,False,State,Rajasthan,Youth,Club League,Male
4,70600,2094,R-LEAGUE YOUTH U13 (Boy's) 2025-26,"19 June 2026,",\nPoornima University,NaN,True,This match has finished,2,Navya Mittal,...,418674,19665,1,False,False,State,Rajasthan,Youth,Club League,Male


In [22]:
male_youth_club_player_matches['match_id'].value_counts()

match_id
59426    93
58938    86
59425    86
59424    83
58939    82
         ..
50704     8
55520     8
67669     8
50710     6
50745     6
Name: count, Length: 6990, dtype: int64

In [23]:
print("Player-match rows:", len(male_youth_club_player_matches))
print("Unique matches:", male_youth_club_player_matches["match_id"].nunique())
print("Unique players:", male_youth_club_player_matches["player_aiff_id"].nunique())
print("Unique competitions:", male_youth_club_player_matches["competition_name"].nunique())

Player-match rows: 306894
Unique matches: 6990
Unique players: 33790
Unique competitions: 139


In [24]:
# Any duplicated player-match combinations?
male_youth_club_player_matches.duplicated(
    subset=["match_id", "player_aiff_id"]
).sum()

np.int64(2)

In [25]:
# 1. How many player rows are missing an AIFF ID?
print(
    male_youth_club_player_matches["player_aiff_id"]
    .isna()
    .value_counts(dropna=False)
)

print(
    "Missing AIFF ID rate:",
    male_youth_club_player_matches["player_aiff_id"].isna().mean()
)

# 0 missing values!

player_aiff_id
False    306894
Name: count, dtype: int64
Missing AIFF ID rate: 0.0


In [26]:
# are there multiple names attached to one aiff id? 

names_per_id = (
    male_youth_club_player_matches
    .dropna(subset=["player_aiff_id"]) #drop null values which are none
    .groupby("player_aiff_id")["player_name"]
    .nunique()
    .sort_values(ascending=False)
)

names_per_id.value_counts().sort_index()

# 33,186 AIFF IDs (98.1%) are associated with exactly one player name ✅
# 539 AIFF IDs (1.6%) appear with two names
# 62 AIFF IDs (0.18%) appear with three names
# 3 AIFF IDs appear with four names

player_name
1    33186
2      539
3       62
4        3
Name: count, dtype: int64

In [28]:
# for ~600 cases where one aiff id has multiple player names, is it only because of names being written oddly?
problem_ids = names_per_id[names_per_id > 1].index

(
    male_youth_club_player_matches[
        male_youth_club_player_matches["player_aiff_id"].isin(problem_ids)
    ]
    [["player_aiff_id", "player_name", "competition_name", "player_team_id"]]
    .drop_duplicates()
    .sort_values(["player_aiff_id", "player_name"])
)


,player_aiff_id,player_name,competition_name,player_team_id
395541,48902,Rayaan Shakeer,Reliance Foundation Development League 2025-26,11410
85884,48902,Rayaan Shakeer (loc),KSFA Youth Premier League U-21 2024-25,11410
250379,49943,Ahan Gupta,AIFF Elite Youth League 2025-26,11410
19849,49943,Ahan Gupta (loc),KSFA Youth Premier League U-17 Tier-1 2024-25,11410
85832,56789,Sarath Rajkumar (loc),KSFA Youth Premier League U-21 2024-25,12772
...,...,...,...,...
521830,497361,Siddhant Deepak Munoli (loc) P,KSFA Youth Premier League Sub Junior Tier-1 (U...,14401
520459,498090,T K Vihaan,KSFA Youth Premier League Sub Junior Tier-2 (U...,17425
518757,498090,T K Vihaan (loc)*,KSFA Youth Premier League Junior Tier-1 (U-15)...,17425
598007,513252,Gulshan Kh Singha,GFA U-14 BOYS DIV II LEAGUE 2025-26,17368


In [29]:
# can we track one player's match appearances? Count the number of unique matches for each player.
appearances_per_player = (
    male_youth_club_player_matches
    .groupby("player_aiff_id")["match_id"]
    .nunique()
)

appearances_per_player.describe()

count    33790.000000
mean         9.082332
std          7.557275
min          1.000000
25%          3.000000
50%          7.000000
75%         13.000000
max         77.000000
Name: match_id, dtype: float64

In [30]:
appearances_per_player

player_aiff_id
48902     12
49943      6
50005      2
50006      2
51010      5
          ..
529424    14
529545    14
529546    14
529703    14
529901    14
Name: match_id, Length: 33790, dtype: int64

In [33]:
critical_cols = [
    "match_id",
    "competition_id",
    "competition_name",
    "match_date",
    "player_name",
    "player_aiff_id",
    "player_team_id",
    "status",
]

male_youth_club_player_matches[critical_cols].isna().mean().sort_values(ascending=False) * 100

# 0 MISSINGNESS IN CRITICAL FIELDS

match_id            0.0
competition_id      0.0
competition_name    0.0
match_date          0.0
player_name         0.0
player_aiff_id      0.0
player_team_id      0.0
status              0.0
dtype: float64

In [34]:
teams_per_match = (
    male_youth_club_player_matches
    .groupby("match_id")["player_team_id"]
    .nunique()
)

teams_per_match.value_counts().sort_index()

player_team_id
1     334
2    6656
Name: count, dtype: int64